<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='../4.%20authentication_and_security/11.%20security_best_practices.ipynb' style='padding:6px 14px; background:#007bff; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>← Chapter 11: Security</a>
  <a href='../TOC.md' style='padding:6px 14px; background:#6c757d; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>📚 Table of Contents</a>
  <a href='13.%20api_authentication_with_jwt.ipynb' style='padding:6px 14px; background:#28a745; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>Chapter 13: JWT Auth →</a>
</div>

# Chapter 12: Building REST APIs with Flask
## Serving Data, Not Pages

A useful beginner mental model here is to separate the shape of the data from the operations performed on it. Once you know what is being represented and who depends on that representation, the rules become easier to predict.

> *"Not every client is a browser. Mobile apps, React frontends, other services — they all need data, not HTML. A REST API serves JSON instead of rendered pages. Flask is excellent at this: lightweight, fast, and fully in your control."*

> ❓ **What makes an API "RESTful"?** REST uses HTTP verbs semantically: `GET` reads, `POST` creates, `PUT`/`PATCH` updates, `DELETE` removes. Each URL represents a *resource* (e.g., `/users/42`), and the server keeps no state between requests.

## 🎯 The Big Picture

Modern applications rarely live in one place. A **mobile app**, a **React frontend**, a **third-party integration**, a **microservice** — all of these need to talk to your backend. But they don't want HTML. They want **raw data** they can work with.

A **REST API** is a backend that:
- Accepts structured HTTP requests
- Returns structured JSON responses
- Uses HTTP methods (`GET`, `POST`, `PUT`, `DELETE`) to signal intent
- Uses HTTP status codes to signal the outcome

Flask's minimalism makes it perfect for building REST APIs. There's no template rendering, no form handling — just routes that return JSON. Clean, fast, and predictable.

```
Browser/App → HTTP Request → Flask Route → JSON Response → Browser/App
```

## 🧠 Core Concepts: The "Why"

The easiest way to read this section is to keep one plain-language question in view: what is The "Why" actually responsible for? Once that job is clear, the terminology stops feeling arbitrary and the details start to line up.

### What Is REST?

**REST** (Representational State Transfer) is an architectural style for building APIs. Key ideas:

| Constraint | Plain English |
|---|---|
| **Stateless** | Each request contains all info needed — no server-side session |
| **Uniform Interface** | Resources accessed via consistent URLs + HTTP verbs |
| **Client-Server** | Frontend and backend are decoupled |
| **Cacheable** | Responses can declare themselves cacheable |

### Resources + Verbs = REST

API and integration material becomes easier when you treat it as agreement design rather than just endpoint syntax. Watch the shape of the data, the guarantees around it, and the failure cases.

| HTTP Verb | URL | Action | SQL Equivalent |
|---|---|---|---|
| `GET` | `/posts` | List all posts | `SELECT *` |
| `GET` | `/posts/1` | Get one post | `SELECT WHERE id=1` |
| `POST` | `/posts` | Create a new post | `INSERT` |
| `PUT` | `/posts/1` | Replace a post | `UPDATE` |
| `PATCH` | `/posts/1` | Partially update | `UPDATE (partial)` |
| `DELETE` | `/posts/1` | Delete a post | `DELETE` |
### The Waiter Analogy

A first read goes better when you connect every new idea to a concrete responsibility and a concrete use case. Once those anchors are in place, the terminology in this section becomes much easier to reuse accurately.

> 🍽️ **A REST API is a waiter who speaks JSON.**
> You sit down (make a request), order in a structured format (HTTP + JSON body), and get structured data back (JSON response). The kitchen (database) doesn't care who you are — every order is self-contained.
### HTTP Status Codes

If this topic is new, keep translating each new term into a simple question: what job does it do, when would you reach for it, and what would you confuse it with if you were moving too quickly? That habit makes the section easier to retain.

| Code | Meaning | When to use |
|---|---|---|
| `200` | OK | Successful GET, PUT, DELETE |
| `201` | Created | Successful POST |
| `204` | No Content | Successful DELETE (nothing to return) |
| `400` | Bad Request | Client sent invalid data |
| `401` | Unauthorized | Not logged in |
| `403` | Forbidden | Logged in but no permission |
| `404` | Not Found | Resource doesn't exist |
| `422` | Unprocessable Entity | Data format correct but content invalid |
| `500` | Internal Server Error | Something crashed on the server |

## ⚡ Syntax & First Usage

If this topic is new, keep translating each new term into a simple question: what job does it do, when would you reach for it, and what would you confuse it with if you were moving too quickly? That habit makes the section easier to retain.

### `jsonify()` — Turning Python into JSON

If this topic is new, keep translating each new term into a simple question: what job does it do, when would you reach for it, and what would you confuse it with if you were moving too quickly? That habit makes the section easier to retain.

Flask's `jsonify()` serialises Python objects to JSON and sets `Content-Type: application/json` automatically.

In [ ]:
# Minimal REST API example
from flask import Flask, jsonify, request

app = Flask(__name__)

# In-memory "database" for demonstration
posts_db = [
    {"id": 1, "title": "Hello Flask", "body": "My first post"},
    {"id": 2, "title": "REST APIs", "body": "Building APIs is fun"},
]

@app.route('/api/posts', methods=['GET'])
def get_posts():
    return jsonify(posts_db), 200

@app.route('/api/posts/<int:post_id>', methods=['GET'])
def get_post(post_id):
    post = next((p for p in posts_db if p['id'] == post_id), None)
    if post is None:
        return jsonify({"error": "Post not found"}), 404
    return jsonify(post), 200

print("Routes defined:")
print("GET /api/posts        -> list all posts")
print("GET /api/posts/<id>   -> get one post")

### Reading JSON from the Request Body

The key to this section is sequence. If you can explain what happens first, what happens next, and where control moves after that, the moving parts here become much easier to reason about.



In [ ]:
from flask import Flask, jsonify, request

app = Flask(__name__)
posts_db = []

@app.route('/api/posts', methods=['POST'])
def create_post():
    data = request.get_json()  # Returns None if Content-Type isn't application/json

    if not data:
        return jsonify({"error": "Request body must be JSON"}), 400

    if 'title' not in data or 'body' not in data:
        return jsonify({"error": "Missing required fields: title, body"}), 422

    new_post = {
        "id": len(posts_db) + 1,
        "title": data['title'],
        "body": data['body']
    }
    posts_db.append(new_post)
    return jsonify(new_post), 201  # 201 Created, not 200!

print("POST /api/posts -> 201 on success")
print("Returns 400 if body is not JSON")
print("Returns 422 if required fields are missing")

## 🔬 Deep Dive

If this topic is new, keep translating each new term into a simple question: what job does it do, when would you reach for it, and what would you confuse it with if you were moving too quickly? That habit makes the section easier to retain.

### Complete CRUD API for a `Post` Resource

API and integration material becomes easier when you treat it as agreement design rather than just endpoint syntax. Watch the shape of the data, the guarantees around it, and the failure cases.



In [ ]:
from flask import Flask, jsonify, request

app = Flask(__name__)
posts = {}
next_id = 1

# CREATE
@app.route('/api/v1/posts', methods=['POST'])
def create_post():
    data = request.get_json()
    if not data:
        return jsonify(error="Request body must be JSON"), 400
    if not data.get('title') or not data.get('body'):
        return jsonify(error="Fields 'title' and 'body' are required"), 422
    global next_id
    post = {"id": next_id, "title": data['title'], "body": data['body']}
    posts[next_id] = post
    next_id += 1
    return jsonify(post), 201

# READ ALL
@app.route('/api/v1/posts', methods=['GET'])
def list_posts():
    return jsonify(list(posts.values())), 200

# READ ONE
@app.route('/api/v1/posts/<int:post_id>', methods=['GET'])
def get_post(post_id):
    post = posts.get(post_id)
    if not post:
        return jsonify(error=f"Post {post_id} not found"), 404
    return jsonify(post), 200

# UPDATE
@app.route('/api/v1/posts/<int:post_id>', methods=['PUT'])
def update_post(post_id):
    if post_id not in posts:
        return jsonify(error=f"Post {post_id} not found"), 404
    data = request.get_json()
    if not data:
        return jsonify(error="Request body must be JSON"), 400
    posts[post_id].update({
        "title": data.get('title', posts[post_id]['title']),
        "body":  data.get('body',  posts[post_id]['body']),
    })
    return jsonify(posts[post_id]), 200

# DELETE
@app.route('/api/v1/posts/<int:post_id>', methods=['DELETE'])
def delete_post(post_id):
    if post_id not in posts:
        return jsonify(error=f"Post {post_id} not found"), 404
    deleted = posts.pop(post_id)
    return jsonify({"message": f"Post '{deleted['title']}' deleted"}), 200

print("Complete CRUD API:")
print("POST   /api/v1/posts       -> Create  (201)")
print("GET    /api/v1/posts       -> List    (200)")
print("GET    /api/v1/posts/<id>  -> Get one (200 or 404)")
print("PUT    /api/v1/posts/<id>  -> Update  (200 or 404)")
print("DELETE /api/v1/posts/<id>  -> Delete  (200 or 404)")

### Adding `to_dict()` to SQLAlchemy Models

In real apps, your models should know how to serialise themselves:

In [ ]:
from datetime import datetime

class Post:
    def __init__(self, id, title, body, author_id, created_at=None):
        self.id = id
        self.title = title
        self.body = body
        self.author_id = author_id
        self.created_at = created_at or datetime.utcnow()

    def to_dict(self):
        """Serialise to a JSON-safe dict. Control what gets exposed."""
        return {
            "id": self.id,
            "title": self.title,
            "body": self.body,
            "author_id": self.author_id,
            "created_at": self.created_at.isoformat(),
        }

    def to_summary(self):
        """Lighter version for list views."""
        return {"id": self.id, "title": self.title}

# Usage in a route:
# return jsonify([p.to_dict() for p in Post.query.all()]), 200

post = Post(1, "Hello REST", "Body content", author_id=42)
print(post.to_dict())

### JSON Error Handlers — Return JSON, Not HTML

By default, Flask returns HTML error pages. In an API, all responses must be JSON:

In [ ]:
from flask import Flask, jsonify

app = Flask(__name__)

@app.errorhandler(400)
def bad_request(e):
    return jsonify(error="Bad request", details=str(e)), 400

@app.errorhandler(404)
def not_found(e):
    return jsonify(error="Resource not found", details=str(e)), 404

@app.errorhandler(405)
def method_not_allowed(e):
    return jsonify(error="Method not allowed"), 405

@app.errorhandler(422)
def unprocessable(e):
    return jsonify(error="Unprocessable entity", details=str(e)), 422

@app.errorhandler(500)
def server_error(e):
    return jsonify(error="Internal server error"), 500

print("Without error handlers:")
print("  GET /nonexistent -> <html>404 Not Found...</html>")
print()
print("With error handlers:")
print('  GET /nonexistent -> {"error": "Resource not found"}')

### ⚖️ Plain Flask Routes vs Flask-RESTX Resource Classes

**Flask-RESTX** adds structure, auto-documentation, and request validation on top of Flask. Here's the same endpoint both ways:

In [ ]:
# APPROACH 1: Plain Flask Routes
from flask import Flask, jsonify, request

app = Flask(__name__)

@app.route('/api/posts', methods=['GET', 'POST'])
def posts():
    if request.method == 'GET':
        return jsonify({"posts": []}), 200
    data = request.get_json()
    return jsonify(data), 201

@app.route('/api/posts/<int:post_id>', methods=['GET', 'PUT', 'DELETE'])
def post(post_id):
    if request.method == 'GET':
        return jsonify({"id": post_id}), 200
    if request.method == 'PUT':
        return jsonify({"id": post_id, "updated": True}), 200
    return jsonify({"deleted": True}), 200

print("Plain Flask: one function handles multiple methods via if/elif")
print("Simple, but grows messy as business logic increases")

In [ ]:
# APPROACH 2: Flask-RESTX Resource Classes (pip install flask-restx)

# from flask_restx import Api, Namespace, Resource, fields
# api = Api(app, version='1.0', title='Blog API', doc='/')  # Swagger UI at /
# ns  = api.namespace('posts', description='Post operations')

# post_model = api.model('Post', {
#     'id':    fields.Integer(readonly=True),
#     'title': fields.String(required=True),
#     'body':  fields.String(required=True),
# })

# @ns.route('/')
# class PostList(Resource):
#     @ns.marshal_list_with(post_model)
#     def get(self): """List all posts"""

#     @ns.expect(post_model)
#     @ns.marshal_with(post_model, code=201)
#     def post(self): """Create a new post"""

print("Flask-RESTX benefits:")
print("  * Free Swagger UI at / (auto-generated OpenAPI spec)")
print("  * Request validation via reqparse")
print("  * Response marshalling with marshal_with")
print("  * Each HTTP method is its own clean function")
print("  * Namespaces for organising large APIs")
print()
print("Trade-offs:")
print("  - More boilerplate for simple endpoints")
print("  - Best for large APIs with external consumers")

### ⚖️ `jsonify()` vs Returning a Dict

Since **Flask 1.0**, returning a dict from a view automatically converts it to JSON:

In [ ]:
from flask import Flask, jsonify

app = Flask(__name__)

# Option 1: jsonify() — explicit, works in all Flask versions
@app.route('/api/v1')
def api_v1():
    return jsonify({"version": 1, "status": "ok"}), 200

# Option 2: Return dict directly (Flask 1.0+)
@app.route('/api/v2')
def api_v2():
    return {"version": 2, "status": "ok"}, 200

# Option 3: Return list directly (Flask 2.2+)
@app.route('/api/v3')
def api_v3():
    return [{"id": 1}, {"id": 2}], 200

print("jsonify()     Explicit. Best for custom headers or all Flask versions.")
print("Return dict:  Concise. Flask 1.0+ shorthand.")
print("Return list:  Flask 2.2+ only — check your version before using.")
print()
print("Recommendation: jsonify() is safest for broad compatibility.")

### CORS — Allowing Cross-Origin Requests

Browsers block JavaScript from reading responses from a different origin unless the API explicitly allows it.

In [ ]:
# pip install flask-cors

# Allow all origins (development only — NEVER in production)
# from flask_cors import CORS
# CORS(app)

# Specific origins (production-safe)
# CORS(app, resources={
#     r"/api/*": {
#         "origins": ["https://myapp.com", "https://staging.myapp.com"],
#         "methods": ["GET", "POST", "PUT", "DELETE"],
#         "allow_headers": ["Content-Type", "Authorization"]
#     }
# })

# Per-route decorator
# from flask_cors import cross_origin
# @app.route('/api/public')
# @cross_origin()
# def public_endpoint():
#     return jsonify({"public": True})

print("CORS is needed when:")
print("  - A React/Vue SPA on one domain calls your Flask API on another")
print()
print("CORS is NOT needed when:")
print("  - API and frontend are on the same domain + port")
print("  - Server-to-server communication (no browser involved)")
print()
print("Never use CORS(app) with no restrictions in production.")

### API Versioning — Never Break Existing Clients

API and integration material becomes easier when you treat it as agreement design rather than just endpoint syntax. Watch the shape of the data, the guarantees around it, and the failure cases.



In [ ]:
from flask import Flask, jsonify, Blueprint

app = Flask(__name__)

v1 = Blueprint('v1', __name__, url_prefix='/api/v1')

@v1.route('/posts')
def v1_posts():
    return jsonify({"version": 1, "posts": [{"id": 1, "title": "Old format"}]})

v2 = Blueprint('v2', __name__, url_prefix='/api/v2')

@v2.route('/posts')
def v2_posts():
    return jsonify({
        "version": 2,
        "data": [{"id": 1, "title": "New format"}],
        "meta": {"page": 1, "total": 1}
    })

app.register_blueprint(v1)
app.register_blueprint(v2)

print("URL versioning strategy:")
print("  GET /api/v1/posts  -> old format, kept for backwards compatibility")
print("  GET /api/v2/posts  -> new format with pagination envelope")
print()
print("Rule: Once a version is public, NEVER break it.")
print("      Add new versions instead of modifying existing ones.")

## 🧪 What If?

A first read goes better when you connect every new idea to a concrete responsibility and a concrete use case. Once those anchors are in place, the terminology in this section becomes much easier to reuse accurately.

### What if `request.get_json()` returns `None`?

Whenever a process has several stages, beginners benefit from tracing the handoff between them. Follow the order carefully and notice what information each step receives, transforms, and passes on.



In [ ]:
from flask import Flask, request, jsonify

app = Flask(__name__)

@app.route('/api/test', methods=['POST'])
def test():
    data = request.get_json()  # Returns None if Content-Type != application/json

    if data is None:
        data = request.get_json(force=True)  # Try ignoring Content-Type

    if data is None:
        return jsonify(error="Could not parse body as JSON. "
                             "Set Content-Type: application/json"), 400
    return jsonify(received=data), 200

print("force=True  -> parse regardless of Content-Type header")
print("silent=True -> return None instead of raising on parse errors")
print()
print("The client must include:")
print("  -H 'Content-Type: application/json'")

In [ ]:
# What if you return 200 for a resource creation?
print("Wrong: POST /api/posts -> 200 OK")
print("  Client can't tell if the resource was created or just fetched.")
print()
print("Correct: POST /api/posts -> 201 Created")
print("  Response body: the newly created resource")
print("  Optional header: Location: /api/posts/42")
print()
print("This matters for HTTP caches and REST-compliant API consumers.")

In [ ]:
# What if CORS is not set up and a React frontend calls your API?
print("Without CORS, browser blocks the response with:")
print()
print("  Access to fetch at 'http://localhost:5000/api/posts' from origin")
print("  'http://localhost:3000' has been blocked by CORS policy:")
print("  No 'Access-Control-Allow-Origin' header is present.")
print()
print("Important: The REQUEST was sent; the server responded normally.")
print("The BROWSER blocked JavaScript from READING the response.")
print()
print("Fix:")
print("  from flask_cors import CORS")
print("  CORS(app, origins=['http://localhost:3000'])")

## 🚀 Real-World Project Link

A first read goes better when you connect every new idea to a concrete responsibility and a concrete use case. Once those anchors are in place, the terminology in this section becomes much easier to reuse accurately.

**Our Blog's REST API Layer**

The Blog application exposes a REST API at `/api/v1/` so mobile clients and future React frontends can consume data:

```
GET    /api/v1/posts              -> Paginated list (public)
GET    /api/v1/posts/<id>         -> Single post (public)
POST   /api/v1/posts              -> Create post (requires auth)
PUT    /api/v1/posts/<id>         -> Update post (requires ownership)
DELETE /api/v1/posts/<id>         -> Delete post (requires ownership)
```

```python
# blog/api/posts.py
from . import api_bp

@api_bp.route('/posts')
def list_posts():
    page = request.args.get('page', 1, type=int)
    posts = Post.query.order_by(Post.created_at.desc()).paginate(page=page, per_page=10)
    return jsonify({
        "posts": [p.to_dict() for p in posts.items],
        "total": posts.total,
        "pages": posts.pages,
        "current_page": posts.page
    })
```

## 📋 Chapter Summary & Cheat Sheet

### Key Takeaways

1. **REST APIs serve JSON, not HTML** — `jsonify()` is your primary tool
2. **HTTP methods are verbs** — GET (read), POST (create), PUT (update), DELETE (delete)
3. **Status codes matter** — 200 OK, 201 Created, 400 Bad Request, 404 Not Found, 500 Error
4. **Always validate `request.get_json()`** — returns `None` if `Content-Type` is wrong
5. **Return JSON error responses** — override Flask's default HTML error pages
6. **Flask-RESTX** adds structure + free Swagger UI for larger APIs
7. **Flask-CORS** required when API is called from a different domain in a browser

### Cheat Sheet

If this topic is new, keep translating each new term into a simple question: what job does it do, when would you reach for it, and what would you confuse it with if you were moving too quickly? That habit makes the section easier to retain.

```python
# Install
pip install flask flask-restx flask-cors

# Return JSON
return jsonify(data), 200          # explicit
return {"key": "value"}, 200       # Flask 1.0+ shorthand

# Read request JSON
data = request.get_json()           # None if wrong Content-Type
data = request.get_json(force=True) # ignore Content-Type header

# Status codes
return jsonify(post), 201           # Created
return jsonify(error="..."), 400    # Bad Request
return jsonify(error="..."), 404    # Not Found
return '', 204                      # No Content (DELETE)

# JSON error handlers
@app.errorhandler(404)
def not_found(e):
    return jsonify(error=str(e)), 404

# CORS
from flask_cors import CORS
CORS(app, origins=["https://myapp.com"])

# Flask-RESTX
from flask_restx import Api, Resource, fields
api = Api(app, doc='/')
class PostList(Resource):
    def get(self): ...
    def post(self): ...
api.add_resource(PostList, '/posts')
```

## 💪 Practice Prompts

A first read goes better when you connect every new idea to a concrete responsibility and a concrete use case. Once those anchors are in place, the terminology in this section becomes much easier to reuse accurately.

1. **Build a Tags API**: Add a `Tag` resource to the Blog API. A post can have many tags. Build `GET /api/v1/tags`, `GET /api/v1/posts/<id>/tags`, `POST /api/v1/posts/<id>/tags`. Return proper status codes and JSON error responses.

2. **Paginate the posts endpoint**: Modify `GET /api/v1/posts` to accept `?page=1&per_page=10` query parameters. Return a response envelope with `data`, `page`, `per_page`, `total`, and `pages` fields. Return 400 if `per_page > 100`.

3. **Convert to Flask-RESTX**: Rewrite your CRUD API using Flask-RESTX `Namespace`, `Resource`, and `fields` models. Add request parsers with validation. Verify the auto-generated Swagger UI appears at `/`.

<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='../4.%20authentication_and_security/11.%20security_best_practices.ipynb' style='padding:6px 14px; background:#007bff; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>← Chapter 11: Security</a>
  <a href='../TOC.md' style='padding:6px 14px; background:#6c757d; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>📚 Table of Contents</a>
  <a href='13.%20api_authentication_with_jwt.ipynb' style='padding:6px 14px; background:#28a745; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>Chapter 13: JWT Auth →</a>
</div>